# Baseline Model Training Walkthrough

This notebook demonstrates training the baseline YOLOv8n fire detection model.

In [ ]:
import os
import sys
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add parent directory to path
sys.path.append('..')

from src.datasets import FireDetectionDataset, collate_fn
from src.models import create_baseline_model
from torch.utils.data import DataLoader

%matplotlib inline

## 1. Load Configuration

In [ ]:
# Load baseline configuration
with open('../configs/baseline.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration:")
print(yaml.dump(config, default_flow_style=False))

## 2. Create Datasets and DataLoaders

In [ ]:
# Training dataset
train_dataset = FireDetectionDataset(
    img_dir=config['data']['train_img_dir'],
    label_dir=config['data']['train_label_dir'],
    img_size=config['model']['img_size'],
    augment=True
)

# Validation dataset
val_dataset = FireDetectionDataset(
    img_dir=config['data']['val_img_dir'],
    label_dir=config['data']['val_label_dir'],
    img_size=config['model']['img_size'],
    augment=False
)

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config['training']['batch_size'],
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config['training']['batch_size'],
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn
)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

## 3. Create Model

In [ ]:
# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Create baseline model
model = create_baseline_model(
    num_classes=config['model']['num_classes'],
    pretrained=config['model']['pretrained'],
    device=device
)

# Print model info
model_info = model.get_model_info()
print("\nModel Information:")
for key, value in model_info.items():
    print(f"  {key}: {value}")

## 4. Training (Using Ultralytics YOLO)

For actual training, we use the Ultralytics YOLO training pipeline which is optimized and well-tested.

In [ ]:
# Training using Ultralytics (simplified)
# In practice, use: python scripts/train.py --config configs/baseline.yaml

print("""\nTo train the model, run:

    python scripts/train.py --config configs/baseline.yaml

This will:
1. Load the configuration
2. Create datasets and dataloaders
3. Initialize the model and optimizer
4. Train for specified epochs with validation
5. Save checkpoints and best model
6. Log metrics to TensorBoard
""")

## 5. Monitor Training

You can monitor training progress using TensorBoard:

In [ ]:
# Load TensorBoard (run this in a separate cell or terminal)
# %load_ext tensorboard
# %tensorboard --logdir runs/

print("""\nTo view training progress:

In terminal:
    tensorboard --logdir runs/

Then open: http://localhost:6006
""")

## 6. Visualize Sample Predictions (After Training)

In [ ]:
# This cell assumes you have a trained model checkpoint
# checkpoint_path = 'runs/baseline_*/best_model.pt'

# Load trained model
# model.load_checkpoint(checkpoint_path)
# model.eval()

# Get a batch from validation set
# images, targets = next(iter(val_loader))
# images = images.to(device)

# Make predictions
# with torch.no_grad():
#     predictions = model.predict(images)

# Visualize results
# (Implementation depends on prediction format)

print("After training, use this section to visualize predictions.")

## 7. Next Steps

After training the baseline model:

1. **Evaluate** on test set: `python scripts/evaluate.py --checkpoint runs/baseline_*/best_model.pt --config configs/baseline.yaml`

2. **Try improved models**: Train V1-V5 variants with different strategies

3. **Run ablation study**: Compare all models systematically

4. **Test robustness**: Evaluate under challenging conditions

5. **Export for deployment**: Convert to ONNX/TensorRT for production